<a href="https://colab.research.google.com/github/contreras-juan/procesamiento_lenguaje_natural/blob/main/Cuadernos/18_Aplicaciones_Preentrenados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color: #FECB05; text-align: center;">Aplicaciones de los modelos preentrenados</h1>

<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)

---

<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [<span style="color: #005C99;">Introducción</span>](#introduccion)
- [<span style="color: #005C99;">Clasificación de texto</span>](#clasificacion-de-texto)
- [<span style="color: #005C99;">Reconocimiento de entidades</span>](#reconocimiento-de-entidades)
- [<span style="color: #005C99;">Traducción automática</span>](#traduccion-automatica)
- [<span style="color: #005C99;">Sumarización</span>](#sumarizacion)
- [<span style="color: #005C99;">Respuesta a preguntas</span>](#respuesta-a-preguntas)
- [<span style="color: #005C99;">Referencias</span>](#referencias)

---

<h2 style="color: #007ACC;">Instalación de librerías</h2>

Ejecuta esta celda si trabajas en Google Colab o en un entorno nuevo.

> **GPU recomendada.** Este cuaderno descarga y ejecuta modelos de cientos de millones de parámetros. En Google Colab activa una GPU en *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*. El código también funciona en CPU, pero mucho más lento.

> **Sobre el framework.** Desde la versión 5, la librería `transformers` de Hugging Face solo funciona con **PyTorch** (abandonó TensorFlow y JAX). Por eso, a partir de este cuaderno los modelos preentrenados se cargan con PyTorch; las ideas son las mismas que en Keras y casi todo el código usa la interfaz de alto nivel de `transformers`.

In [1]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib seaborn scikit-learn datasets "transformers>=5" accelerate torch nltk sentencepiece sacremoses rouge_score

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import transformers
from transformers import AutoTokenizer, pipeline, set_seed
from transformers import AutoModelForSeq2SeqLM, AutoModelForTokenClassification, AutoModelForQuestionAnswering
from datasets import load_dataset

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
set_seed(SEED)  # fija las semillas de Python, NumPy y PyTorch
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print(f"transformers {transformers.__version__} | PyTorch {torch.__version__} | dispositivo: {dispositivo}")

transformers 5.20.2 | PyTorch 2.11.0+cu128 | dispositivo: cuda


<a id="introduccion"></a>
<h2 style="color: #007ACC;">Introducción</h2>

En el cuaderno 17 ajustamos BETO a una tarea propia. Muchas veces ni siquiera eso es necesario: en el Hugging Face Hub hay miles de modelos **ya ajustados** para tareas concretas, publicados por universidades, empresas y la comunidad. En este cuaderno usamos modelos de ese tipo para las tareas clásicas del PLN que recorrimos en el curso, y los **evaluamos** con datos en español, comparándolos con lo que construimos en módulos anteriores:

| Tarea | Modelo preentrenado | Comparación con el curso |
|---|---|---|
| Clasificación (estrellas, *zero-shot*) | BERT multilingüe, mDeBERTa-NLI | Taller 05 (1–5 estrellas con RNN) |
| Reconocimiento de entidades | BETO ajustado en CoNLL-2002 | Taller 04 (MEMM) |
| Traducción | MarianMT (OPUS-MT) | Cuaderno 14 (Seq2Seq con GRU) |
| Sumarización | mT5 ajustado en XL-Sum | Línea base: primera oración |
| Respuesta a preguntas | BETO ajustado en SQuAD-es | — |

Una advertencia antes de empezar: un modelo publicado **no es un modelo validado para tu problema**. Hay que leer su tarjeta (con qué datos se entrenó, licencia, limitaciones) y evaluarlo con datos propios, como haremos aquí.

<a id="clasificacion-de-texto"></a>
<h2 style="color: #007ACC;">Clasificación de texto</h2>

**Calificación de reseñas.** El modelo `nlptown/bert-base-multilingual-uncased-sentiment` es un BERT multilingüe ajustado para predecir de 1 a 5 estrellas en reseñas de productos en seis idiomas, entre ellos el español. Evaluémoslo en la tarea del taller 05, con 1 000 reseñas de prueba del corpus de Amazon:

In [3]:
resenas = load_dataset("SetFit/amazon_reviews_multi_es", split="test").shuffle(seed=SEED).select(range(1000))
estrellas_reales = np.array(resenas["label"]) + 1  # la etiqueta va de 0 a 4

estrellas = pipeline("text-classification", model="nlptown/bert-base-multilingual-uncased-sentiment",
                     device=dispositivo)
inicio = time.perf_counter()
salidas = estrellas(list(resenas["text"]), batch_size=64, truncation=True, max_length=256)
print(f"Tiempo: {time.perf_counter() - inicio:.1f} s para 1 000 reseñas")
print("Ejemplo de salida:", salidas[0])

estrellas_pred = np.array([int(s["label"][0]) for s in salidas])
pd.Series({
    "exactitud": np.mean(estrellas_pred == estrellas_reales),
    "exactitud ±1 estrella": np.mean(np.abs(estrellas_pred - estrellas_reales) <= 1),
    "error absoluto medio (MAE)": np.mean(np.abs(estrellas_pred - estrellas_reales)),
}, name="nlptown en 1 000 reseñas de prueba").round(3).to_frame()

Tiempo: 5.2 s para 1 000 reseñas
Ejemplo de salida: {'label': '5 stars', 'score': 0.5943552255630493}


,nlptown en 1 000 reseñas de prueba
exactitud,0.549
exactitud ±1 estrella,0.915
error absoluto medio (MAE),0.562


In [4]:
pd.crosstab(pd.Series(estrellas_reales, name="real"), pd.Series(estrellas_pred, name="predicha"))

predicha,1,2,3,4,5
real,,,,,
1,138,34,15,1,1
2,54,92,46,6,5
3,15,52,89,37,7
4,1,7,44,104,41
5,5,7,15,58,126


Sin ningún entrenamiento de nuestra parte, el modelo supera a los del taller 05: 0,55 de exactitud y un MAE de 0,56, frente a 0,51 del mejor modelo del taller 05 en prueba. La matriz de confusión muestra el mismo patrón que allí: los extremos (1 y 5 estrellas) se reconocen bien y las calificaciones intermedias se confunden con sus vecinas; el 91 % de las predicciones está a una estrella o menos de la real.

Cuidado con la conclusión: según su tarjeta, este modelo se ajustó con reseñas de productos, y es posible que haya visto reseñas de este mismo corpus de Amazon. Cuando un modelo se evalúa con datos que pudo ver en su entrenamiento hablamos de **contaminación**, y su resultado puede ser optimista. Es un problema frecuente con modelos públicos y con los LLM, cuyos datos de entrenamiento rara vez se conocen por completo.

**Clasificación *zero-shot*.** ¿Y si no hay ningún modelo para nuestras categorías ni datos etiquetados? Un modelo de **inferencia en lenguaje natural** (NLI) decide si una *premisa* implica una *hipótesis*. Se puede usar para clasificar con cualquier conjunto de etiquetas: la premisa es el texto y la hipótesis es *"Este ejemplo trata sobre {etiqueta}."*; la etiqueta con mayor probabilidad de implicación gana (Yin et al., 2019). Usaremos mDeBERTa-v3, ajustado en NLI en 15 idiomas.

In [5]:
zero_shot = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", device=dispositivo)

mensajes = [
    "Me cobraron dos veces la factura de este mes y nadie me responde.",
    "El pedido lleva dos semanas en tránsito y el número de seguimiento no funciona.",
    "La licuadora dejó de funcionar al tercer uso, el motor huele a quemado.",
    "No puedo entrar a mi cuenta, dice que mi contraseña es incorrecta.",
    "¿Tienen tiendas físicas en Tunja o solo venden por internet?",
]
categorias = ["facturación", "envío", "producto defectuoso", "acceso a la cuenta", "información general"]
filas = []
for m in mensajes:
    r = zero_shot(m, candidate_labels=categorias, hypothesis_template="Este mensaje trata sobre {}.")
    filas.append({"mensaje": m, "categoría": r["labels"][0], "probabilidad": round(r["scores"][0], 3),
                  "segunda opción": r["labels"][1]})
pd.set_option("display.max_colwidth", 90)
pd.DataFrame(filas)

,mensaje,categoría,probabilidad,segunda opción
0,Me cobraron dos veces la factura de este mes y nadie me responde.,facturación,0.998,acceso a la cuenta
1,El pedido lleva dos semanas en tránsito y el número de seguimiento no funciona.,envío,0.895,acceso a la cuenta
2,"La licuadora dejó de funcionar al tercer uso, el motor huele a quemado.",producto defectuoso,0.426,información general
3,"No puedo entrar a mi cuenta, dice que mi contraseña es incorrecta.",acceso a la cuenta,0.969,producto defectuoso
4,¿Tienen tiendas físicas en Tunja o solo venden por internet?,acceso a la cuenta,0.568,envío


El modelo acierta en los cuatro primeros mensajes, aunque con poca confianza en el de la licuadora (0,43), y falla en el último: una pregunta de información general queda como *acceso a la cuenta*, con una probabilidad baja (0,57). Las categorías se pueden cambiar sin reentrenar nada, pero sus nombres y la plantilla de la hipótesis influyen mucho en el resultado y conviene probar varias. Midamos cuánto rinde frente a un modelo entrenado con datos etiquetados, en el sentimiento binario de los cuadernos 11 a 17:

In [6]:
binario = load_dataset("SetFit/amazon_reviews_multi_es", split="test").filter(lambda r: r["label"] != 2)
binario = binario.shuffle(seed=SEED).select(range(1000))
y_bin = (np.array(binario["label"]) >= 3).astype(int)

inicio = time.perf_counter()
salidas = zero_shot(list(binario["text"]), candidate_labels=["positiva", "negativa"],
                    hypothesis_template="La opinión sobre el producto es {}.", batch_size=32)
pred_bin = np.array([s["labels"][0] == "positiva" for s in salidas]).astype(int)
print(f"Exactitud zero-shot (sentimiento binario, 1 000 reseñas): {np.mean(pred_bin == y_bin):.3f} "
      f"({time.perf_counter() - inicio:.0f} s)")

Exactitud zero-shot (sentimiento binario, 1 000 reseñas): 0.862 (7 s)


Sin un solo ejemplo etiquetado, la clasificación *zero-shot* alcanza una exactitud de 0,86, comparable a la de TF-IDF entrenado con 5 000 reseñas (0,87) pero por debajo de BETO ajustado con esas reseñas en el cuaderno 17 (0,93). Su costo es otro: el modelo NLI se ejecuta **una vez por cada etiqueta candidata**, así que con muchas categorías es lento. En la práctica es una excelente forma de **arrancar** (o de etiquetar datos de forma preliminar) antes de tener un conjunto etiquetado.

<a id="reconocimiento-de-entidades"></a>
<h2 style="color: #007ACC;">Reconocimiento de entidades</h2>

En el taller 04 entrenamos un MEMM para reconocer entidades (personas, lugares, organizaciones y otras) en el corpus CoNLL-2002, con un F1 por entidades de 0,77 en prueba. Evaluemos ahora un BETO ajustado para NER que está publicado en el Hub, con la misma partición de prueba (`esp.testb`) y la misma métrica.

Los modelos de clasificación de tokens trabajan con **subpalabras**, pero CoNLL etiqueta **palabras**. Al tokenizar con `is_split_into_words=True`, el tokenizador recuerda a qué palabra pertenece cada subpalabra (`word_ids()`), y tomamos la etiqueta de la **primera subpalabra** de cada palabra:

In [7]:
import nltk
from nltk.corpus import conll2002

_ = nltk.download("conll2002", quiet=True)


def cargar(particion):
    return [[(w, ner) for w, _, ner in s] for s in conll2002.iob_sents(particion)]


entrenamiento, validacion, prueba = cargar("esp.train"), cargar("esp.testa"), cargar("esp.testb")
print(f"Oraciones: entrenamiento {len(entrenamiento)} | validación {len(validacion)} | prueba {len(prueba)}")


def etiquetar(oraciones, modelo, tok, lote=32):
    resultado = []
    for i in range(0, len(oraciones), lote):
        palabras = [[w for w, _ in s] for s in oraciones[i:i + lote]]
        entrada = tok(palabras, is_split_into_words=True, truncation=True, max_length=512,
                      padding=True, return_tensors="pt")
        with torch.no_grad():
            pred = modelo(**entrada.to(modelo.device)).logits.argmax(-1).cpu().numpy()
        for j, ws in enumerate(palabras):
            etiquetas, vistas = ["O"] * len(ws), set()
            for k, w_id in enumerate(entrada.word_ids(j)):
                if w_id is not None and w_id not in vistas:  # primera subpalabra de cada palabra
                    vistas.add(w_id)
                    etiquetas[w_id] = modelo.config.id2label[int(pred[j, k])]
            resultado.append(etiquetas)
    return resultado


nombre_ner = "mrm8488/bert-spanish-cased-finetuned-ner"
tok_ner = AutoTokenizer.from_pretrained(nombre_ner)
modelo_hub = AutoModelForTokenClassification.from_pretrained(nombre_ner).to(dispositivo).eval()
print("Etiquetas del modelo:", modelo_hub.config.id2label)
pred_hub = etiquetar(prueba, modelo_hub, tok_ner)
print(list(zip([w for w, _ in prueba[2]], pred_hub[2])))

Oraciones: entrenamiento 8323 | validación 1915 | prueba 1517


[transformers] BertForTokenClassification LOAD REPORT from: mrm8488/bert-spanish-cased-finetuned-ner
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 
Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Etiquetas del modelo: {0: 'B-LOC', 1: 'B-MISC', 2: 'B-ORG', 3: 'B-PER', 4: 'I-LOC', 5: 'I-MISC', 6: 'I-ORG', 7: 'I-PER', 8: 'O'}


[('Las', 'O'), ('reservas', 'O'), ('"', 'O'), ('on', 'O'), ('line', 'O'), ('"', 'O'), ('de', 'O'), ('billetes', 'O'), ('aéreos', 'O'), ('a', 'O'), ('través', 'O'), ('de', 'O'), ('Internet', 'B-MISC'), ('aumentaron', 'O'), ('en', 'O'), ('España', 'B-LOC'), ('un', 'O'), ('300', 'O'), ('por', 'O'), ('ciento', 'O'), ('en', 'O'), ('el', 'O'), ('primer', 'O'), ('trimestre', 'O'), ('de', 'O'), ('este', 'O'), ('año', 'O'), ('con', 'O'), ('respecto', 'O'), ('al', 'O'), ('mismo', 'O'), ('período', 'O'), ('de', 'O'), ('1999', 'O'), (',', 'O'), ('aseguró', 'O'), ('hoy', 'O'), ('Iñigo', 'B-PER'), ('García', 'I-PER'), ('Aranda', 'I-PER'), (',', 'O'), ('responsable', 'O'), ('de', 'O'), ('comunicación', 'O'), ('de', 'O'), ('Savia', 'B-PER'), ('Amadeus', 'I-PER'), ('.', 'O')]


Evaluamos **por entidades completas** con la función del taller 04: una entidad cuenta como correcta solo si coinciden su inicio, su fin y su tipo.

In [8]:
def entidades(tags):
    ents, tipo, inicio = set(), None, None
    for i, t in enumerate(list(tags) + ["O"]):
        continua = t.startswith("I-") and t[2:] == tipo
        if not continua:
            if tipo is not None:
                ents.add((inicio, i, tipo))
                tipo = None
            if t != "O":
                tipo, inicio = t[2:], i
    return ents


def evaluar_entidades(reales, predichas):
    tp, n_pred, n_real = Counter(), Counter(), Counter()
    for r, p in zip(reales, predichas):
        er, ep = entidades(r), entidades(p)
        for _, _, tipo in ep:
            n_pred[tipo] += 1
        for _, _, tipo in er:
            n_real[tipo] += 1
        for _, _, tipo in er & ep:
            tp[tipo] += 1

    def prf(tp_, np_, nr_):
        prec = tp_ / np_ if np_ else 0.0
        rec = tp_ / nr_ if nr_ else 0.0
        return prec, rec, (2 * prec * rec / (prec + rec) if prec + rec else 0.0)

    total = prf(sum(tp.values()), sum(n_pred.values()), sum(n_real.values()))
    por_tipo = {t: prf(tp[t], n_pred[t], n_real[t])[2] for t in sorted(n_real)}
    return {"precisión": total[0], "recall": total[1], "F1": total[2], **{f"F1 {t}": f for t, f in por_tipo.items()}}


reales = [[t for _, t in s] for s in prueba]
tabla_ner = pd.DataFrame({"BETO-NER del Hub": evaluar_entidades(reales, pred_hub)}).T
tabla_ner.round(3)

,precisión,recall,F1,F1 LOC,F1 MISC,F1 ORG,F1 PER
BETO-NER del Hub,0.978,0.983,0.98,0.977,0.963,0.979,0.995


¿F1 de 0,98? Es **demasiado bueno para ser cierto**: el artículo de BETO reporta un F1 de 0,884 en esta misma tarea, el mejor sistema de la competencia CoNLL-2002 obtuvo 0,81, y la propia tarjeta del modelo reporta 0,90. La tarjeta explica la diferencia: el autor **juntó todo el corpus CoNLL en español y lo dividió al azar 80/20**, así que la mayoría de las oraciones de `esp.testb` estaban en su conjunto de entrenamiento. Es un caso claro de **contaminación**: el modelo puede ser bueno, pero esta cifra no mide su capacidad de generalizar.

Para tener una comparación honesta con el MEMM, ajustamos nosotros mismos BETO para NER, respetando la partición oficial: `esp.train` para entrenar, `esp.testa` para validar y `esp.testb` solo para la evaluación final.

La única novedad respecto a la clasificación de textos (cuaderno 17) es **alinear las etiquetas con las subpalabras**: la primera subpalabra de cada palabra recibe la etiqueta de la palabra, y las demás subpalabras y los tokens especiales reciben `-100`, el valor que PyTorch ignora en la pérdida.

In [9]:
from datasets import Dataset
from transformers import DataCollatorForTokenClassification, Trainer, TrainingArguments

nombres_etiquetas = sorted({t for s in entrenamiento for _, t in s})
etiqueta2id = {t: i for i, t in enumerate(nombres_etiquetas)}
tok_beto = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")


def a_dataset_ner(oraciones):
    palabras = [[w for w, _ in s] for s in oraciones]
    entrada = tok_beto(palabras, is_split_into_words=True, truncation=True, max_length=256)
    etiquetas = []
    for j, s in enumerate(oraciones):
        fila, anterior = [], None
        for w_id in entrada.word_ids(j):
            fila.append(-100 if w_id is None or w_id == anterior else etiqueta2id[s[w_id][1]])
            anterior = w_id
        etiquetas.append(fila)
    return Dataset.from_dict({**entrada, "labels": etiquetas})


ds_train, ds_val = a_dataset_ner(entrenamiento), a_dataset_ner(validacion)
ejemplo = ds_train[2]
print(list(zip(tok_beto.convert_ids_to_tokens(ejemplo["input_ids"])[:12], ejemplo["labels"][:12])))

[('[CLS]', -100), ('El', 8), ('Abogado', 3), ('General', 7), ('del', 7), ('Estado', 7), (',', 8), ('Dar', 3), ('##yl', -100), ('W', 7), ('##illiams', -100), (',', 8)]


In [10]:
set_seed(SEED)
modelo_ner = AutoModelForTokenClassification.from_pretrained(
    "dccuchile/bert-base-spanish-wwm-cased", num_labels=len(nombres_etiquetas),
    id2label=dict(enumerate(nombres_etiquetas)), label2id=etiqueta2id)

argumentos = TrainingArguments(output_dir="checkpoints/beto-ner", learning_rate=5e-5, num_train_epochs=3,
                               per_device_train_batch_size=16, per_device_eval_batch_size=64,
                               weight_decay=0.01, warmup_steps=0.1, eval_strategy="epoch",
                               save_strategy="no", logging_strategy="epoch",
                               fp16=torch.cuda.is_available(), report_to="none", seed=SEED)
entrenador = Trainer(model=modelo_ner, args=argumentos, train_dataset=ds_train, eval_dataset=ds_val,
                     data_collator=DataCollatorForTokenClassification(tok_beto))
inicio = time.perf_counter()
entrenador.train()
print(f"Tiempo de fine-tuning: {time.perf_counter() - inicio:.0f} s")

modelo_ner.eval()
print(f"F1 en validación (esp.testa): {evaluar_entidades([[t for _, t in s] for s in validacion], etiquetar(validacion, modelo_ner, tok_beto))['F1']:.3f}")
tabla_ner.loc["BETO ajustado por nosotros (esp.train)"] = pd.Series(evaluar_entidades(reales, etiquetar(prueba, modelo_ner, tok_beto)))
tabla_ner.loc["MEMM (taller 04)", "F1"] = 0.77
tabla_ner.round(3)

[transformers] BertForTokenClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 
Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.188021,0.077074
2,0.034805,0.080164
3,0.016595,0.084501


Tiempo de fine-tuning: 140 s


F1 en validación (esp.testa): 0.861


,precisión,recall,F1,F1 LOC,F1 MISC,F1 ORG,F1 PER
BETO-NER del Hub,0.978,0.983,0.980,0.977,0.963,0.979,0.995
BETO ajustado por nosotros (esp.train),0.861,0.882,0.871,0.867,0.708,0.869,0.960
MEMM (taller 04),NaN,NaN,0.770,NaN,NaN,NaN,NaN


Con la partición oficial, nuestro BETO alcanza un F1 de **0,871** en prueba (0,861 en validación): muy por encima del MEMM del taller 04 (0,77) y del mejor sistema de la competencia CoNLL-2002 (0,81), y cerca del 0,884 reportado por Cañete et al. (2020). Es 11 puntos menos que la cifra contaminada. Las personas son casi perfectas (F1 0,96) y las entidades `MISC` (obras, eventos, nacionalidades...) siguen siendo las más difíciles (0,71), por ser la categoría más heterogénea.

**Lección:** antes de confiar en la métrica de un modelo publicado, hay que averiguar con qué datos se entrenó. Si la tarjeta no lo dice, no se puede descartar la contaminación.

Probemos con un texto colombiano, fuera del dominio de noticias españolas de 2002. El `pipeline` de `token-classification` con `aggregation_strategy="simple"` agrupa las subpalabras y los tokens B/I en entidades completas:

In [11]:
ner = pipeline("token-classification", model=modelo_ner, tokenizer=tok_beto, aggregation_strategy="simple",
               device=dispositivo)
texto = ("Gustavo Petro se reunió en Tunja con el gobernador de Boyacá y con directivos de la UPTC "
         "para hablar del Festival Internacional de la Cultura y de la vía al Llano.")
pd.DataFrame(ner(texto))[["word", "entity_group", "score"]].round(3)

,word,entity_group,score
0,Gustavo Petro,PER,0.998
1,Tunja,LOC,0.968
2,Boyacá,LOC,0.881
3,UPTC,ORG,0.950
4,Festival Internacional de la Cultura,MISC,0.998
5,Llano,LOC,0.651


<a id="traduccion-automatica"></a>
<h2 style="color: #007ACC;">Traducción automática</h2>

Los modelos **OPUS-MT** de la Universidad de Helsinki (Tiedemann & Thottingal, 2020) son Transformers codificador-decodificador (arquitectura MarianMT) entrenados con los corpus paralelos de [OPUS](https://opus.nlpl.eu/). Hay más de mil, uno por par de idiomas, de unos 75 millones de parámetros cada uno.

En la versión 5 de `transformers`, la traducción y el resumen ya no tienen un `pipeline` propio: se usan directamente el tokenizador, `AutoModelForSeq2SeqLM` y su método `generate`, que implementa la búsqueda en haz del cuaderno 14.

In [12]:
tok_mt = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-es-en")
traductor = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-es-en").to(dispositivo).eval()


def traducir(textos, lote=64, haz=4):
    resultado = []
    for i in range(0, len(textos), lote):
        entrada = tok_mt(textos[i:i + lote], return_tensors="pt", padding=True, truncation=True).to(dispositivo)
        with torch.no_grad():
            salida = traductor.generate(**entrada, num_beams=haz, max_new_tokens=128)
        resultado += tok_mt.batch_decode(salida, skip_special_tokens=True)
    return resultado


for frase in ["El perro duerme en el sofá.", "No me gusta tener que levantarme temprano los lunes.",
              "Ayer fui a la tienda, pero estaba cerrada, así que volví a casa."]:
    print(f"{frase}\n   → {traducir([frase])[0]}")

El perro duerme en el sofá.
   → The dog sleeps on the couch.


No me gusta tener que levantarme temprano los lunes.
   → I don't like having to get up early on Mondays.


Ayer fui a la tienda, pero estaba cerrada, así que volví a casa.
   → I went to the store yesterday, but it was closed, so I went home.


Comparemos con el Seq2Seq con GRU del cuaderno 14 en **las mismas 500 oraciones de prueba** del corpus Tatoeba, con el mismo BLEU (calculado con NLTK sobre el texto en minúsculas y con la puntuación separada). Reconstruimos la partición exactamente como en el cuaderno 14:

In [13]:
import random
import re
import urllib.request
import zipfile
from io import BytesIO
from nltk.translate.bleu_score import corpus_bleu

URL = "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
with zipfile.ZipFile(BytesIO(urllib.request.urlopen(URL).read())) as z:
    lineas = z.read("spa-eng/spa.txt").decode("utf-8").strip().split("\n")
pares = [tuple(linea.split("\t")[:2])[::-1] for linea in lineas]
pares_cortos = [(es, en) for es, en in pares if len(es.split()) <= 10 and len(en.split()) <= 10]
random.Random(SEED).shuffle(pares_cortos)
muestra = pares_cortos[:60_000][-2_000:][:500]  # las primeras 500 oraciones de prueba del cuaderno 14


def tokens_en(texto):
    """La misma normalización del cuaderno 14: minúsculas y puntuación como tokens."""
    texto = re.sub(r"([¿?¡!.,])", r" \1 ", texto.lower())
    return re.sub(r"[^a-záéíóúüñ¿?¡!.,' \[\]]", "", texto).split()


inicio = time.perf_counter()
hipotesis = traducir([es for es, _ in muestra])
t_mt = time.perf_counter() - inicio
bleu_opus = corpus_bleu([[tokens_en(en)] for _, en in muestra], [tokens_en(h) for h in hipotesis])
pd.DataFrame({"BLEU": {"Seq2Seq GRU, haz k=4 (cuaderno 14)": 0.362, "OPUS-MT, haz k=4": bleu_opus}}).round(3)

,BLEU
"Seq2Seq GRU, haz k=4 (cuaderno 14)",0.362
"OPUS-MT, haz k=4",0.661


In [14]:
pd.set_option("display.max_colwidth", 70)
pd.DataFrame({"español": [es for es, _ in muestra[:10]], "referencia": [en for _, en in muestra[:10]],
              "OPUS-MT": hipotesis[:10]})

,español,referencia,OPUS-MT
0,Tuvimos un debate muy vigoroso.,We had a very vigorous debate.,We had a very vigorous debate.
1,No debería haberte besado.,I shouldn't have kissed you.,I shouldn't have kissed you.
2,"Me vuelvo a casa, ya me he hartado de Tom.",I'm going home. I've had enough of Tom.,"I'm going home, I've had enough of Tom."
3,Estamos cómodos en nuestra nueva casa.,We are comfortable in our new house.,We're comfortable in our new home.
4,Él es soltero todavía.,He's still single.,He's still single.
5,Mi mamá se levanta temprano por la mañana.,My mother gets up early in the morning.,My mom gets up early in the morning.
6,He estado preparándome.,I've been preparing.,I've been getting ready.
7,Tom pensó que era injusto.,Tom thought it was unfair.,Tom thought it was unfair.
8,No pueden despedirles.,They can't fire you.,They can't be fired.
9,Me alegro de que vengas.,I'm glad you're coming.,I'm glad you're here.


OPUS-MT casi duplica el BLEU del Seq2Seq del cuaderno 14 (0,66 frente a 0,36). Sus diferencias con la referencia suelen ser paráfrasis válidas (*I'm* / *I am*, *mustn't* / *shouldn't*), que BLEU castiga.

Dos advertencias sobre esta comparación:

- **No es justa en recursos:** OPUS-MT se entrenó con cientos de millones de pares de oraciones y una arquitectura Transformer con atención; el Seq2Seq, con 56 000 pares en CPU.
- **Contaminación:** Tatoeba forma parte de OPUS, así que es probable que el modelo haya visto muchas de estas oraciones (o casi idénticas) durante su entrenamiento.

Donde la diferencia se nota más es en oraciones largas, que el Seq2Seq sin atención no podía traducir:

In [15]:
parrafo = ("La Universidad Pedagógica y Tecnológica de Colombia, con sede principal en Tunja, ofrece programas "
           "de pregrado y posgrado en varias ciudades de Boyacá. Sus estudiantes participan en proyectos de "
           "investigación sobre inteligencia artificial aplicada a la agricultura y al medio ambiente.")
print(traducir([parrafo])[0])

The Pedagogical and Technological University of Colombia, headquartered in Tunja, offers undergraduate and postgraduate programs in several cities of Boyacá. Its students participate in research projects on artificial intelligence applied to agriculture and the environment.


<a id="sumarizacion"></a>
<h2 style="color: #007ACC;">Sumarización</h2>

La **sumarización** puede ser:

- **extractiva**: se seleccionan las oraciones más importantes del texto original;
- **abstractiva**: se genera un resumen nuevo, con palabras que pueden no estar en el texto. Es una tarea Seq2Seq.

Usaremos **mT5** (la versión multilingüe de T5, Xue et al., 2021) ajustado en **XL-Sum** (Hasan et al., 2021), un corpus de artículos de la BBC en 44 idiomas en el que el resumen de referencia es la frase introductoria que escribe el periodista. Evaluamos en 100 artículos de BBC Mundo del conjunto de prueba.

In [16]:
xlsum = load_dataset("parquet", data_files={
    "test": "hf://datasets/csebuetnlp/xlsum@refs/convert/parquet/spanish/test/0000.parquet"})["test"]
xlsum = xlsum.shuffle(seed=SEED).select(range(100))
print(f"Palabras por artículo: mediana {np.median([len(t.split()) for t in xlsum['text']]):.0f} | "
      f"por resumen: mediana {np.median([len(t.split()) for t in xlsum['summary']]):.0f}")
print("\nTítulo:", xlsum[0]["title"])
print("Resumen de referencia:", xlsum[0]["summary"])
print("Inicio del artículo:", xlsum[0]["text"][:400], "...")

Palabras por artículo: mediana 667 | por resumen: mediana 24
Título: Marcelo Odebrecht, el empresario que creó un imperio de la construcción y hoy está en el centro del escándalo de corrupción que sacude a los gobiernos de América Latina
Resumen de referencia: El apellido Odebrecht ha quedado irremediablemente atado a uno de los escándalos de corrupción más grandes en la historia de América Latina.
Inicio del artículo: Marcelo Odebrecht fue condenado a 19 años de prisión. Y en medio de todo el huracán, está la figura de Marcelo Odebrecht, un ejecutivo de 47 años, que pasó de ser el gerente de una de las firmas de construcción más grandes del mundo, a tener temblando a gobernantes y mercados de la región con sus confesiones de corrupción a las autoridades judiciales brasileñas. Pero hasta 2015, antes de que se de ...


In [17]:
nombre_sum = "csebuetnlp/mT5_multilingual_XLSum"
tok_sum = AutoTokenizer.from_pretrained(nombre_sum)
resumidor = AutoModelForSeq2SeqLM.from_pretrained(nombre_sum).to(dispositivo).eval()
print(f"Parámetros: {resumidor.num_parameters():,}")


def resumir(textos, lote=8):
    resultado = []
    for i in range(0, len(textos), lote):
        textos_limpios = [re.sub(r"\s+", " ", t) for t in textos[i:i + lote]]
        entrada = tok_sum(textos_limpios, return_tensors="pt", padding=True, truncation=True,
                          max_length=512).to(dispositivo)
        with torch.no_grad():
            salida = resumidor.generate(**entrada, max_new_tokens=84, num_beams=4, no_repeat_ngram_size=2)
        resultado += tok_sum.batch_decode(salida, skip_special_tokens=True)
    return resultado


inicio = time.perf_counter()
resumenes = resumir(list(xlsum["text"]))
print(f"Tiempo: {time.perf_counter() - inicio:.0f} s para 100 artículos")
for i in range(3):
    print(f"\n[{i}] Referencia: {xlsum[i]['summary']}\n    Modelo:     {resumenes[i]}")

Parámetros: 582,401,280


Tiempo: 30 s para 100 artículos
[0] Referencia: El apellido Odebrecht ha quedado irremediablemente atado a uno de los escándalos de corrupción más grandes en la historia de América Latina.
    Modelo:     La tormenta tropical que sacudió a Brasil en el último mes de agosto ha dejado al menos siete muertos y cientos de heridos.
[1] Referencia: A los 40 años del Foro Económico Mundial, BBC Mundo se propuso explorar algunas de sus particularidades.
    Modelo:     El Foro Económico Mundial (WEF, por sus siglas en inglés) es uno de los eventos más importantes de la historia reciente de Davos, Suiza. Su edición plenaria se celebra este miércoles en la ciudad suiza de Zurich, y algunos de sus principales temas son claves para la celebración.
[2] Referencia: Se trata de un evento tan extraño que es apenas el segundo que se documenta en la historia de la neonatología.
    Modelo:     Fue la primera vez que se identifican gemelos del mismo sexo durante el embarazo.


La métrica estándar es **ROUGE** (Lin, 2004), que mide el solapamiento con el resumen de referencia: ROUGE-1 y ROUGE-2 cuentan unigramas y bigramas comunes, y ROUGE-L la subsecuencia común más larga. Es el análogo de BLEU orientado al *recall*.

El tokenizador por defecto de la librería `rouge_score` elimina los caracteres no ingleses (las tildes y la ñ), así que le pasamos uno propio. Como línea base usamos la **primera oración** del artículo (*lead-1*), una base extractiva difícil de superar en noticias:

In [18]:
from rouge_score import rouge_scorer


class TokenizadorEspanol:
    def tokenize(self, texto):
        return re.findall(r"\w+", texto.lower())


evaluador = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], tokenizer=TokenizadorEspanol())


def rouge(predicciones, referencias):
    puntajes = [evaluador.score(r, p) for p, r in zip(predicciones, referencias)]
    return {m: np.mean([s[m].fmeasure for s in puntajes]) for m in ["rouge1", "rouge2", "rougeL"]}


def primera_oracion(texto):
    return re.split(r"(?<=[.!?])\s+", texto.strip(), maxsplit=1)[0]


pd.DataFrame({
    "primera oración (lead-1)": rouge([primera_oracion(t) for t in xlsum["text"]], xlsum["summary"]),
    "mT5 XL-Sum": rouge(resumenes, xlsum["summary"]),
}).T.round(3)

,rouge1,rouge2,rougeL
primera oración (lead-1),0.194,0.047,0.152
mT5 XL-Sum,0.302,0.117,0.243


El modelo abstractivo supera a la primera oración en las tres métricas (ROUGE-1 de 0,30 frente a 0,19), en línea con lo que reportan Hasan et al. (2021) para el español. Las referencias de XL-Sum son frases introductorias escritas por periodistas, que no repiten la primera oración del artículo; por eso *lead-1* es aquí una base débil.

Pero los ejemplos muestran el riesgo de la generación: para el artículo sobre **Marcelo Odebrecht**, el modelo produjo un resumen sobre *una tormenta tropical que dejó siete muertos en Brasil*, algo que el artículo no menciona (solo usa *huracán* y *tormenta* en sentido figurado). Comprobamos que el resultado es el mismo al resumir ese artículo solo, sin lote: no es un error del código sino una **alucinación** del modelo. En el segundo ejemplo, sitúa el Foro de Davos en Zúrich. Un resumen abstractivo puede introducir **información que no está en el texto**. Una medida sencilla es la proporción de bigramas del resumen que no aparecen en el artículo (bigramas *novedosos*): son necesarios para parafrasear, pero también son el lugar donde aparecen las **alucinaciones**.

In [19]:
def bigramas(texto):
    tokens = re.findall(r"\w+", texto.lower())
    return set(zip(tokens, tokens[1:]))


novedad = [len(bigramas(r) - bigramas(t)) / max(len(bigramas(r)), 1) for r, t in zip(resumenes, xlsum["text"])]
novedad_ref = [len(bigramas(r) - bigramas(t)) / max(len(bigramas(r)), 1) for r, t in zip(xlsum["summary"], xlsum["text"])]
print(f"Bigramas novedosos — modelo: {np.mean(novedad):.1%} | referencias humanas: {np.mean(novedad_ref):.1%}")

i = int(np.argmax(novedad))
print(f"\nResumen con más bigramas novedosos ({novedad[i]:.0%}):\n  {resumenes[i]}")
print(f"Referencia:\n  {xlsum[i]['summary']}")

Bigramas novedosos — modelo: 45.1% | referencias humanas: 75.7%
Resumen con más bigramas novedosos (90%):
  El ejército iraquí continúa su ofensiva contra los militantes del grupo yihadista Estado Islámico de Irak (ISIS, por sus siglas en inglés).
Referencia:
  Fuertes combates continúan en Irak entre fuerzas del gobierno y militantes del grupo radical Estado Islámico de Irak y el Levante, ISIS por sus siglas en inglés.


El 45 % de los bigramas de los resúmenes del modelo y el 76 % de los de las referencias humanas no están en el artículo: XL-Sum se construyó con las frases introductorias de la BBC, que a menudo añaden contexto ausente del cuerpo de la noticia. Un modelo entrenado para imitarlas aprende también a **añadir información**, y es difícil distinguir cuándo esa información es correcta. Por eso los resúmenes automáticos de documentos importantes (médicos, legales) requieren verificación. Lo retomamos en el cuaderno 19.

<a id="respuesta-a-preguntas"></a>
<h2 style="color: #007ACC;">Respuesta a preguntas</h2>

En la **respuesta a preguntas extractiva** (*extractive QA*), el modelo recibe una pregunta y un texto (el *contexto*) y devuelve el fragmento del contexto que responde la pregunta. Es la tarea del corpus SQuAD (Rajpurkar et al., 2016).

La arquitectura es sencilla: se concatenan pregunta y contexto (`[CLS] pregunta [SEP] contexto [SEP]`), se pasan por BERT y una capa lineal produce, para cada token, dos puntajes: ser el **inicio** y ser el **fin** de la respuesta. La respuesta es el par (inicio, fin) con mayor puntaje total, con inicio ≤ fin.

Usaremos un BETO destilado (más pequeño y rápido) ajustado en **SQuAD-es**, una traducción automática al español de SQuAD 2.0. SQuAD 2.0 incluye preguntas **sin respuesta** en el contexto; para esas, el modelo debe apuntar al token `[CLS]`.

Como la versión 5 de `transformers` no incluye un `pipeline` de QA, implementamos la extracción:

In [20]:
nombre_qa = "mrm8488/distill-bert-base-spanish-wwm-cased-finetuned-spa-squad2-es"
tok_qa = AutoTokenizer.from_pretrained(nombre_qa)
modelo_qa = AutoModelForQuestionAnswering.from_pretrained(nombre_qa).to(dispositivo).eval()


def responder(pregunta, contexto, max_largo=30):
    entrada = tok_qa(pregunta, contexto, truncation="only_second", max_length=384,
                     return_offsets_mapping=True, return_tensors="pt")
    offsets = entrada.pop("offset_mapping")[0].tolist()
    es_contexto = np.array(entrada.sequence_ids(0)) == 1
    with torch.no_grad():
        salida = modelo_qa(**entrada.to(dispositivo))
    ini, fin = salida.start_logits[0].cpu().numpy(), salida.end_logits[0].cpu().numpy()

    mejor, puntaje = (0, 0), -np.inf
    candidatos_ini = [i for i in np.argsort(ini)[::-1][:20] if es_contexto[i]]
    candidatos_fin = [j for j in np.argsort(fin)[::-1][:20] if es_contexto[j]]
    for i in candidatos_ini:
        for j in candidatos_fin:
            if i <= j < i + max_largo and ini[i] + fin[j] > puntaje:
                mejor, puntaje = (i, j), ini[i] + fin[j]
    puntaje_nulo = ini[0] + fin[0]  # apuntar a [CLS] = "no hay respuesta"
    respuesta = contexto[offsets[mejor[0]][0]:offsets[mejor[1]][1]]
    return respuesta, float(puntaje), float(puntaje_nulo)


contexto = (
    "Tunja es la capital del departamento de Boyacá, en Colombia. Está ubicada sobre la cordillera Oriental "
    "a 2820 metros sobre el nivel del mar y es una de las capitales más altas del país. Fue fundada el 6 de "
    "agosto de 1539 por el capitán Gonzalo Suárez Rendón sobre el antiguo asentamiento muisca de Hunza. "
    "En la ciudad tiene su sede principal la Universidad Pedagógica y Tecnológica de Colombia (UPTC)."
)
for pregunta in ["¿Quién fundó Tunja?", "¿A qué altura está Tunja?", "¿Cómo se llamaba el asentamiento muisca?",
                 "¿Qué universidad tiene su sede en Tunja?", "¿Cuántos habitantes tiene Tunja?"]:
    r, p, nulo = responder(pregunta, contexto)
    print(f"{pregunta:<45} → {r!r:<45} (puntaje {p:5.1f} | sin respuesta {nulo:5.1f})")

[transformers] BertForQuestionAnswering LOAD REPORT from: mrm8488/distill-bert-base-spanish-wwm-cased-finetuned-spa-squad2-es
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 
Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


¿Quién fundó Tunja?                           → 'capitán Gonzalo Suárez Rendón'               (puntaje  13.1 | sin respuesta  -2.0)
¿A qué altura está Tunja?                     → '2820 metros sobre el nivel del mar'          (puntaje  11.5 | sin respuesta  -3.0)
¿Cómo se llamaba el asentamiento muisca?      → 'Hunza'                                       (puntaje  12.1 | sin respuesta  -3.1)
¿Qué universidad tiene su sede en Tunja?      → 'Universidad Pedagógica y Tecnológica de Colombia' (puntaje  12.4 | sin respuesta  -2.4)
¿Cuántos habitantes tiene Tunja?              → 'del'                                         (puntaje   2.0 | sin respuesta  -1.6)


La última pregunta no tiene respuesta en el texto. El modelo devuelve de todos modos el fragmento más plausible, pero su puntaje es mucho menor que el de las demás y queda cerca del puntaje "sin respuesta". Comparar ambos puntajes con un umbral es como se decide **abstenerse** en SQuAD 2.0.

Evaluemos con **SQAC** (Gutiérrez-Fandiño et al., 2021), un corpus de preguntas escritas por personas **en español** (no traducidas) sobre textos de Wikipedia y noticias. Las métricas de SQuAD son:

- **Coincidencia exacta** (EM): la respuesta normalizada (minúsculas, sin puntuación ni artículos) es idéntica a la de referencia.
- **F1**: solapamiento de palabras entre la respuesta y la referencia, que da crédito parcial.

In [21]:
import json
import string
from huggingface_hub import hf_hub_download

with open(hf_hub_download("PlanTL-GOB-ES/SQAC", "test.json", repo_type="dataset"), encoding="utf-8") as f:
    sqac = json.load(f)["data"]
preguntas = [(qa["question"], parrafo["context"], [a["text"] for a in qa["answers"]])
             for articulo in sqac for parrafo in articulo["paragraphs"] for qa in parrafo["qas"]]
random.Random(SEED).shuffle(preguntas)
preguntas = preguntas[:500]
print(f"Preguntas evaluadas: {len(preguntas)} | ejemplo: {preguntas[0][0]} → {preguntas[0][2]}")


def normalizar(texto):
    texto = "".join(c for c in texto.lower() if c not in set(string.punctuation + "¿¡«»"))
    texto = re.sub(r"\b(el|la|los|las|un|una|unos|unas)\b", " ", texto)
    return " ".join(texto.split())


def f1(prediccion, referencia):
    p, r = normalizar(prediccion).split(), normalizar(referencia).split()
    comunes = sum((Counter(p) & Counter(r)).values())
    if comunes == 0:
        return 0.0
    precision, recall = comunes / len(p), comunes / len(r)
    return 2 * precision * recall / (precision + recall)


inicio = time.perf_counter()
resultados = []
for pregunta, contexto, respuestas in preguntas:
    r, _, _ = responder(pregunta, contexto)
    resultados.append({"pregunta": pregunta, "referencia": respuestas[0], "modelo": r,
                       "EM": max(normalizar(r) == normalizar(a) for a in respuestas),
                       "F1": max(f1(r, a) for a in respuestas)})
resultados = pd.DataFrame(resultados)
print(f"Tiempo: {time.perf_counter() - inicio:.0f} s")
resultados[["EM", "F1"]].mean().round(3).to_frame("SQAC (500 preguntas)")

Preguntas evaluadas: 500 | ejemplo: ¿Cuántos sumergibles han apartado? → ['once']


Tiempo: 19 s


,SQAC (500 preguntas)
EM,0.466
F1,0.680


In [22]:
resultados[resultados.F1 < 1].head(10)

,pregunta,referencia,modelo,EM,F1
0,¿Cuántos sumergibles han apartado?,once,once submarinos,False,0.666667
1,¿Qué efecto tendrían las bombas?,esparcir por territorio enemigo las más terribles plagas,"las tropas propias sólo tendrían que avanzar, venciendo la escasa ...",False,0.000000
4,¿Cómo se adapta la piel a las variaciones térmicas?,mediante sus mecanismos de regulación de la temperatura,mediante sus mecanismos de regulación de la temperatura (mayor o m...,False,0.666667
5,¿Cómo actúan estos grupos según la representante de la diplomacia ...,al margen de la ley,no actúan en consecuencia con sus actividades,False,0.000000
7,¿Quién fue la esposa de Maximiliano Hernández Martínez?,Concepción Monteagudo,Petronila Martínez.,False,0.000000
9,¿Qué distingue a la banda?,sus armonías vocales,armonías vocales,False,0.800000
12,¿Quién se cree que escribió el corpus hipocrático?,sus estudiantes y discípulos,hasta diecinueve autores diferentes,False,0.000000
13,¿Quién es el segundo de Uribe?,Francisco Santos Calderón,Francisco Santos,False,0.800000
15,¿Cuándo se enfrentará Arambulet con Gamboa?,el próximo 20 de agosto,20 de agosto,False,0.857143
17,¿En cuánto ha incrementado el número de tigres en India?,20%,manera considerable durante los últimos años,False,0.000000


El modelo obtiene una coincidencia exacta de 0,47 y un F1 de 0,68: responde bien a una parte grande de las preguntas, aunque por debajo de lo que logra en SQuAD-es, con el que se ajustó: las preguntas de SQAC están escritas de forma natural en español, mientras que las de SQuAD-es son traducciones automáticas, con otro estilo. Entre los errores hay respuestas **casi correctas** con límites distintos a los de la referencia (más o menos palabras), que el F1 castiga solo en parte y la coincidencia exacta castiga por completo.

La QA extractiva tiene una limitación de fondo: necesita que alguien le dé el contexto. Los asistentes basados en LLM responden de forma **generativa**, a partir de lo que memorizaron o de documentos recuperados automáticamente (*Retrieval-Augmented Generation*, RAG), con el riesgo de que la respuesta no esté respaldada por ninguna fuente. Es el tema del cuaderno 19.

**Resumen:**

| Tarea | Modelo | Resultado | Referencia del curso |
|---|---|---|---|
| Estrellas (1–5) | BERT multilingüe (nlptown) | Exactitud 0,55 (posible contaminación) | Taller 05: 0,51 |
| Sentimiento binario sin etiquetas | mDeBERTa NLI (*zero-shot*) | Exactitud 0,86 | Cuaderno 17: BETO ajustado, 0,93 |
| NER | BETO ajustado en `esp.train` | F1 0,87 (0,98 el modelo contaminado) | Taller 04: MEMM, F1 0,77 |
| Traducción es→en | OPUS-MT | BLEU 0,66 (posible contaminación) | Cuaderno 14: BLEU 0,36 |
| Resumen | mT5 XL-Sum | ROUGE-1 0,30, con alucinaciones | Primera oración: 0,19 |
| QA | BETO destilado, SQuAD-es | EM 0,47, F1 0,68 en SQAC | — |

Con modelos preentrenados, tareas que requerían semanas de ingeniería de rasgos se resuelven en unas líneas de código. El trabajo del especialista se desplaza a **elegir el modelo adecuado, evaluarlo con datos propios y entender sus fallas y riesgos**.

<a id="referencias"></a>
<h2 style="color: #007ACC;">Referencias</h2>

- Tunstall, L., von Werra, L. & Wolf, T. (2022). *Natural Language Processing with Transformers*. O'Reilly Media.
- Jurafsky, D. & Martin, J. H. *Speech and Language Processing* (3.ª ed. borrador). [web.stanford.edu/~jurafsky/slp3](https://web.stanford.edu/~jurafsky/slp3/)
- Tiedemann, J. & Thottingal, S. (2020). OPUS-MT — Building open translation services for the World. *EAMT*.
- Xue, L. et al. (2021). mT5: A Massively Multilingual Pre-trained Text-to-Text Transformer. *NAACL*.
- Hasan, T. et al. (2021). XL-Sum: Large-Scale Multilingual Abstractive Summarization for 44 Languages. *Findings of ACL*.
- Lin, C.-Y. (2004). ROUGE: A Package for Automatic Evaluation of Summaries. *ACL Workshop*.
- Rajpurkar, P., Jia, R. & Liang, P. (2018). Know What You Don't Know: Unanswerable Questions for SQuAD. *ACL*.
- Gutiérrez-Fandiño, A. et al. (2022). MarIA: Spanish Language Models. *Procesamiento del Lenguaje Natural*, 68 (corpus SQAC).
- Yin, W., Hay, J. & Roth, D. (2019). Benchmarking Zero-shot Text Classification. *EMNLP*.
- He, P., Gao, J. & Chen, W. (2023). DeBERTaV3. *ICLR*.
- Cañete, J. et al. (2020). Spanish Pre-Trained BERT Model and Evaluation Data. *PML4DC at ICLR*.
- Tjong Kim Sang, E. F. (2002). Introduction to the CoNLL-2002 Shared Task. *CoNLL*.
- Rothman, D. (2022). *Transformers for Natural Language Processing* (2.ª ed.). Packt.